# 02 — Grouped Split, Baseline Benchmarking & Model Training

**Project:** FlyRank Ranking Signal Analysis — Predicting Continuous Search Visibility  
**Pipeline Stage:** Notebook 02 of 03 (Model Training & Benchmark Optimization)  
**Target Variable:** Continuous Search Visibility $\log(1 + \text{impressions\_90d})$ on Held-Out Enterprise Client Portfolios  
**Standardized Split:** Grouped Holdout (`GroupShuffleSplit`, `test_size=0.20`, `random_state=42`) with Zero Client Domain Overlap  

---

### Notebook Architecture & Objectives
This notebook executes the training and optimization of our high-accuracy machine learning architecture:
1. **Grouped Holdout Split Design:** Enforce a strict zero-client-leakage partition on `client_id` with programmatic disjointness assertions.
2. **Transparent Heuristic Baselines:** Fit Baseline 1 (Mean Dummy Regressor) and Baseline 2 (Univariate Word Count Rule) to establish the floor to beat.
3. **High-Accuracy Model Training:** 
   - Fit a cross-validated **Polynomial Interaction Ridge Model** (`RidgeCV`) capturing multi-signal continuous synergies.
   - Fit a tuned **HistGradientBoosting Regressor** capturing non-linear thresholds and step-function splits.
   - Construct our unified **Champion Stacking Ensemble** ($0.50 \cdot \text{Ridge} + 0.50 \cdot \text{HGBR}$), which maximizes predictive capacity on unseen client domains.
4. **Payload Export:** Export model predictions, test set ground truth, and fitted pipelines to `work/outputs/model_evaluation_payload.pkl` for evaluation in Notebook 03.

## 1. Grouped Holdout Split Design (Zero Client Leakage)

### 1.1 The Multi-Tenant Generalization Challenge
In multi-tenant SaaS platforms, standard random row splits produce severe **client leakage**: content items from the same client share identical domain authority, root backlink velocity, CMS infrastructure, and brand equity. A random split allows models to memorize client-level baseline volume, inflating test performance while collapsing on brand-new client domains.

### 1.2 The Client Holdout Standard
To guarantee 100% scientific validity:
* We partition strictly by `client_id` using `GroupShuffleSplit(n_splits=1, test_size=0.20, random_state=42)`.
* We assert that training clients and test clients are mutually exclusive:
  $$\text{set}(\text{clients}_{\text{train}}) \cap \text{set}(\text{clients}_{\text{test}}) = \emptyset$$
* Test metrics reflect true generalization performance on entirely unseen enterprise domains.

In [5]:
# Code: Setup, imports, load prepared features, and grouped split
import os
import sys
import pickle
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
from sklearn.model_selection import GroupShuffleSplit
from sklearn.linear_model import LinearRegression, RidgeCV
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from sklearn.impute import SimpleImputer
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
from scipy.stats import spearmanr

RANDOM_STATE = 42

OUTPUTS_DIR = os.path.join('work', 'outputs')
if not os.path.exists(OUTPUTS_DIR):
    OUTPUTS_DIR = os.path.join('..', '..', 'work', 'outputs')

prepared_file = os.path.join(OUTPUTS_DIR, 'prepared_features.parquet')
if not os.path.exists(prepared_file):
    prepared_file = os.path.join(OUTPUTS_DIR, 'prepared_features.csv')
    df = pd.read_csv(prepared_file)
else:
    df = pd.read_parquet(prepared_file)

print(f"[DATA LOAD] Loaded prepared features: {df.shape[0]:,} rows x {df.shape[1]} columns")

# Exclude metadata and target columns from feature matrix X
non_feature_cols = ['content_id', 'client_id', 'target_log_impressions', 'impressions_90d', 'word_count', 'avg_position', 'days_since_last_update', 'content_type', 'main_intent']
feature_cols = [c for c in df.columns if c not in non_feature_cols]

X = df[feature_cols]
y = df['target_log_impressions']
groups = df['client_id']

# Execute Strict Grouped Holdout Split
gss = GroupShuffleSplit(n_splits=1, test_size=0.20, random_state=RANDOM_STATE)
train_idx, test_idx = next(gss.split(X, y, groups=groups))

X_train, X_test = X.iloc[train_idx].copy(), X.iloc[test_idx].copy()
y_train, y_test = y.iloc[train_idx].copy(), y.iloc[test_idx].copy()
clients_train, clients_test = groups.iloc[train_idx], groups.iloc[test_idx]

# Programmatic Leakage Assertion
assert set(clients_train).isdisjoint(set(clients_test)), "CRITICAL LEAKAGE: Client overlap detected between train and test sets!"
print(f"[SPLIT AUDIT] Zero client overlap confirmed between {clients_train.nunique()} train clients and {clients_test.nunique()} test clients.")
print(f"  -> Train Matrix Shape: {X_train.shape} ({len(train_idx):,} rows)")
print(f"  -> Test Matrix Shape: {X_test.shape} ({len(test_idx):,} rows)")
print(f"  -> Train Target Mean: {y_train.mean():.4f} (std={y_train.std():.4f}) | Test Target Mean: {y_test.mean():.4f} (std={y_test.std():.4f})")

[DATA LOAD] Loaded prepared features: 30,000 rows x 57 columns
[SPLIT AUDIT] Zero client overlap confirmed between 25 train clients and 7 test clients.
  -> Train Matrix Shape: (23837, 48) (23,837 rows)
  -> Test Matrix Shape: (6163, 48) (6,163 rows)
  -> Train Target Mean: 6.3089 (std=2.6770) | Test Target Mean: 5.7238 (std=2.6828)


## 2. Transparent Heuristic Baseline Benchmarks

To verify that machine learning provides genuine predictive lift rather than fitting noise, we evaluate two non-ML baselines on the exact same test cohort:
1. **Baseline 1 (Mean Dummy Regressor):** Always predicts the training set target mean $\bar{y}_{\text{train}}$. (Yields $R^2 = 0.0$ on train; negative $R^2$ on held-out clients with a lower mean).
2. **Baseline 2 (Univariate Word Count Rule):** A simple single-variable OLS regression on `log_word_count`, reflecting the common SEO belief that "longer articles rank higher and gain more visibility".

In [6]:
# Code: Fit and evaluate standardized heuristic baselines
def evaluate_metrics(y_true, y_pred, name="Model"):
    r2 = r2_score(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mae = mean_absolute_error(y_true, y_pred)
    try:
        rho, _ = spearmanr(y_true, y_pred)
        if np.isnan(rho):
            rho = 0.0
    except Exception:
        rho = 0.0
    return {
        'Model': name,
        'R2 Score (Higher is better)': round(r2, 4),
        'RMSE (Lower is better)': round(rmse, 4),
        'MAE (Lower is better)': round(mae, 4),
        'Rank Correlation (Spearman rho)': round(rho, 4)
    }

baseline_records = []

# Baseline 1: Mean Dummy
y_pred_dummy = np.full_like(y_test, y_train.mean())
res_dummy = evaluate_metrics(y_test, y_pred_dummy, "Baseline 1: Mean Dummy Regressor")
baseline_records.append(res_dummy)

# Baseline 2: Univariate Word Count Rule
lr_univariate = LinearRegression().fit(X_train[['log_word_count']], y_train)
y_pred_univariate = lr_univariate.predict(X_test[['log_word_count']])
res_univariate = evaluate_metrics(y_test, y_pred_univariate, "Baseline 2: Univariate Rule (log_word_count)")
baseline_records.append(res_univariate)

print("=== Heuristic Baseline Performance on Unseen Test Clients ===")
print(pd.DataFrame(baseline_records).to_string(index=False))

=== Heuristic Baseline Performance on Unseen Test Clients ===
                                       Model  R2 Score (Higher is better)  RMSE (Lower is better)  MAE (Lower is better)  Rank Correlation (Spearman rho)
            Baseline 1: Mean Dummy Regressor                      -0.0476                  2.7457                 2.2757                           0.0000
Baseline 2: Univariate Rule (log_word_count)                      -0.0471                  2.7451                 2.2859                          -0.0571


## 3. High-Accuracy Model Training

### 3.1 Model Architecture Strategy
To achieve the highest honest accuracy without data leakage, we deploy a complementary two-part architecture:
1. **Polynomial Interaction Ridge Regression (`RidgeCV`):**
   * Employs pairwise interaction features ($x_i \times x_j$) across primary engagement and ranking signals.
   * Regularized across 30 cross-validated $\alpha$ penalties (`RidgeCV`, 5-fold CV) to control multi-collinearity.
2. **High-Performance HistGradientBoosting Regressor (`HGBR`):**
   * Builds decision trees sequentially along negative gradients.
   * Captures sharp non-linear thresholds (e.g. position 1 vs 10 step jumps), complex feature interactions, and missing value patterns without matrix explosion.
3. **Champion Blended Ensemble:**
   * A weighted ensemble blending the continuous macro-trends of Polynomial Ridge with the localized threshold splits of Gradient Boosting:
     $$\hat{y}_{\text{Ensemble}} = 0.50 \cdot \hat{y}_{\text{Ridge}} + 0.50 \cdot \hat{y}_{\text{HGBR}}$$

In [7]:
# Code: Train High-Accuracy Models and Ensemble
# 1. Candidate A: Polynomial Interaction Ridge
top_interaction_signals = [
    'log_sessions_90d', 'log_pageviews_90d', 'serp_rank_power', 'log_avg_position',
    'log_word_count', 'log_search_volume', 'log_days_since_last_update',
    'engaged_session_ratio', 'log_users_90d', 'log_engaged_sessions_90d', 'has_rank_data',
    'demand_position_synergy', 'engagement_volume_synergy'
]

poly_ridge_pipe = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('poly', PolynomialFeatures(degree=2, include_bias=False, interaction_only=True)),
    ('scaler', StandardScaler()),
    ('regressor', RidgeCV(alphas=np.logspace(-1, 5, 30), cv=5))
])

print("[TRAINING] Fitting Candidate A: Polynomial Interaction Ridge...")
poly_ridge_pipe.fit(X_train[top_interaction_signals], y_train)
best_alpha = poly_ridge_pipe.named_steps['regressor'].alpha_
y_pred_ridge = poly_ridge_pipe.predict(X_test[top_interaction_signals])
res_ridge = evaluate_metrics(y_test, y_pred_ridge, f"Candidate A: Poly Interaction Ridge (alpha={best_alpha:.2f})")
print(f"  -> RidgeCV Best Alpha: {best_alpha:.2f} | Test R2: {res_ridge['R2 Score (Higher is better)']}")

# 2. Candidate B: HistGradientBoosting Regressor
hgbr_model = HistGradientBoostingRegressor(
    max_iter=350,
    learning_rate=0.03,
    max_leaf_nodes=31,
    min_samples_leaf=25,
    l2_regularization=2.0,
    random_state=RANDOM_STATE
)

print("[TRAINING] Fitting Candidate B: HistGradientBoosting Regressor...")
hgbr_model.fit(X_train, y_train)
y_pred_hgbr = hgbr_model.predict(X_test)
res_hgbr = evaluate_metrics(y_test, y_pred_hgbr, "Candidate B: HistGradientBoosting Regressor")
print(f"  -> HGBR Fitted Successfully | Test R2: {res_hgbr['R2 Score (Higher is better)']}")

# 3. Champion Architecture: Blended Stacking Ensemble
y_pred_ensemble = 0.50 * y_pred_ridge + 0.50 * y_pred_hgbr
res_ensemble = evaluate_metrics(y_test, y_pred_ensemble, "Champion Architecture: Gradient-Boosted & Ridge Ensemble")
print(f"\n[CHAMPION MODEL] Ensemble Test Metrics:")
print(f"  -> R2 Score: {res_ensemble['R2 Score (Higher is better)']} (Explains ~58% of visibility variance)")
print(f"  -> RMSE: {res_ensemble['RMSE (Lower is better)']} (Reduced from 2.7457 baseline)")
print(f"  -> Spearman Rho: {res_ensemble['Rank Correlation (Spearman rho)']} (High monotonic ranking alignment)")

[TRAINING] Fitting Candidate A: Polynomial Interaction Ridge...
  -> RidgeCV Best Alpha: 78.80 | Test R2: 0.5441
[TRAINING] Fitting Candidate B: HistGradientBoosting Regressor...
  -> HGBR Fitted Successfully | Test R2: 0.5467

[CHAMPION MODEL] Ensemble Test Metrics:
  -> R2 Score: 0.578 (Explains ~58% of visibility variance)
  -> RMSE: 1.7426 (Reduced from 2.7457 baseline)
  -> Spearman Rho: 0.7657 (High monotonic ranking alignment)


In [8]:
# Code: Export model payload for evaluation in Notebook 03
payload = {
    'y_test': y_test.values,
    'y_pred_dummy': y_pred_dummy,
    'y_pred_univariate': y_pred_univariate,
    'y_pred_ridge': y_pred_ridge,
    'y_pred_hgbr': y_pred_hgbr,
    'y_pred_ensemble': y_pred_ensemble,
    'test_metadata': df.iloc[test_idx][['content_id', 'client_id', 'target_log_impressions', 'impressions_90d', 'word_count', 'avg_position', 'days_since_last_update', 'content_type', 'main_intent']].copy(),
    'feature_names': feature_cols,
    'top_interaction_signals': top_interaction_signals,
    'best_alpha': best_alpha
}

payload_path = os.path.join(OUTPUTS_DIR, 'model_evaluation_payload.pkl')
with open(payload_path, 'wb') as f:
    pickle.dump(payload, f)

# Also export test predictions as CSV
pred_df = payload['test_metadata'].copy()
pred_df['pred_log_impressions_ensemble'] = y_pred_ensemble
pred_df['pred_log_impressions_ridge'] = y_pred_ridge
pred_df['pred_log_impressions_hgbr'] = y_pred_hgbr
pred_df['pred_impressions_ensemble'] = np.expm1(y_pred_ensemble)
pred_df['residual_gap'] = y_pred_ensemble - pred_df['target_log_impressions']

pred_csv_path = os.path.join(OUTPUTS_DIR, 'model_test_predictions.csv')
pred_df.to_csv(pred_csv_path, index=False)

print(f"[EXPORT] Successfully saved evaluation payload:")
print(f"  -> Pickle: {payload_path} ({os.path.getsize(payload_path):,} bytes)")
print(f"  -> CSV: {pred_csv_path} ({os.path.getsize(pred_csv_path):,} bytes)")
print("[READY] Notebook 02 complete. Ready for evaluation and error audit in Notebook 03.")

[EXPORT] Successfully saved evaluation payload:
  -> Pickle: ..\..\work\outputs\model_evaluation_payload.pkl (828,828 bytes)
  -> CSV: ..\..\work\outputs\model_test_predictions.csv (1,216,148 bytes)
[READY] Notebook 02 complete. Ready for evaluation and error audit in Notebook 03.
